# Segmentation RFM pour le projet GlobalShop Direct

## Préparation 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

In [2]:
df_raw = pd.read_excel("../data/raw/Online_Retail.xlsx")

In [31]:
df_raw.to_csv("../data/raw/online_retail.csv", index=False)

In [32]:
print(df_raw.columns.tolist())
df_raw.head()

['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [33]:
df_raw.shape

(541909, 8)

Le dataset comprend 541 909 lignes et 8 colonnes.

In [34]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


- InvoiceDate est déjà au format date (datetime64).

- 4 colonnes sont de type texte (object), 2 sont de type décimal (float64) et 1 est de type entier (int64).

In [35]:
df_raw.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Deux colonnes ont des valeurs manquantes :
- CustomerID : 135 080 valeurs manquantes.
- Description : 1 454 valeurs manquantes. 

In [36]:
df_raw.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


- **Période couverte** : du 01/12/2010 au 09/12/2011.
- **Quantity** : le minimum est de **-80 995**. 
- **UnitPrice** : le minimum est de **-11 062,06**.
- La moyenne est bien plus élevée que la médiane (Quantity : 9,55 contre 3 ; UnitPrice : 4,61 contre 2,08), à cause de quelques valeurs très grandes.

## 1) Nettoyage 

### 1.1 Suppression des lignes 

In [37]:
df_clean = df_raw.copy()

Lignes_avant = df_clean.shape[0]
df_clean = df_clean.dropna(subset=["CustomerID"])
Lignes_apres = df_clean.shape[0]

print("Lignes avant :", Lignes_avant)
print("Lignes aprés :", Lignes_apres)
print("Lignes supprimées:", Lignes_avant - Lignes_apres)

Lignes avant : 541909
Lignes aprés : 406829
Lignes supprimées: 135080


### 1.2 créer une colonne pour les factures annulées (C)

In [38]:
df_clean["Annulation"] = df_clean["InvoiceNo"].astype(str).str.startswith("C")

print("Lignes d'annulation :", df_clean["Annulation"].sum())
print("Lignes d'achat :", (~df_clean["Annulation"]).sum())

Lignes d'annulation : 8905
Lignes d'achat : 397924


### 1.3 Garder que Quantités et prix positifs (hors annulations)

In [42]:
Lignes_avant = df_clean.shape[0]

df_clean = df_clean[df_clean["UnitPrice"] > 0]
df_clean = df_clean[(df_clean["Quantity"] > 0) | df_clean["Annulation"]]
Lignes_apres = df_clean.shape[0]

print("Lignes avant :", Lignes_avant)
print("Lignes après :", Lignes_apres)
print("Lignes supprimées :", Lignes_avant - Lignes_apres)

Lignes avant : 401564
Lignes après : 401564
Lignes supprimées : 0


### 1.4 Suppression des doublons

In [40]:
Lignes_avant = df_clean.shape[0]
print(" Doublons trouvés :", df_clean.duplicated().sum())

df_clean = df_clean.drop_duplicates()
Lignes_apres = df_clean.shape[0]

print("Lignes avant:", Lignes_avant)
print("Lignes après:", Lignes_apres)
print("Lignes supprimées:", Lignes_avant - Lignes_apres)

 Doublons trouvés : 5225
Lignes avant: 406789
Lignes après: 401564
Lignes supprimées: 5225


### 1.5 Conversion des types (InvoiceDate, CustomerID)

In [44]:
# On vérifie que chaque colonne a le type adapté à son usage : une date pour calculer la récence, un entier pour l'identifiant client
print("Avant :")
print(df_clean[["InvoiceDate", "CustomerID"]].dtypes)

df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])
df_clean["CustomerID"] = df_clean["CustomerID"].astype(int)

print("Après:")
print(df_clean[["InvoiceDate", "CustomerID"]].dtypes)

Avant :
InvoiceDate    datetime64[ns]
CustomerID            float64
dtype: object
Après:
InvoiceDate    datetime64[ns]
CustomerID              int64
dtype: object


- InvoiceDate était déjà au format date (datetime64) : la conversion sert de sécurité, sans rien changer.
- CustomerID passe de décimal (float64) à entier (int64). L'identifiant était stocké en décimal à cause des valeurs manquantes. Une fois celles-ci supprimées, on le repasse en entier pour qu'il reste un identifiant propre, notamment dans les fichiers exportés vers Looker Studio.

### 1.6 Création de la colonne TotalPrice
le fichier ne contient pas le montant de chaque ligne. Il donne seulement la quantité et le prix unitaire.

In [45]:
df_clean["TotalPrice"] = df_clean["Quantity"] * df_clean["UnitPrice"]

print("Montant total des achats :", round(df_clean.loc[~df_clean["Annulation"], "TotalPrice"].sum(), 2))
print("Montant total des annulations :", round(df_clean.loc[df_clean["Annulation"], "TotalPrice"].sum(), 2))
print("Montant net :", round(df_clean["TotalPrice"].sum(), 2))

df_clean[["Quantity", "UnitPrice", "TotalPrice", "Annulation"]].head()

Montant total des achats : 8887208.89
Montant total des annulations : -608689.47
Montant net : 8278519.42


,Quantity,UnitPrice,TotalPrice,Annulation
0,6,2.55,15.30,False
1,6,3.39,20.34,False
2,8,2.75,22.00,False
3,6,3.39,20.34,False
4,6,3.39,20.34,False


In [47]:
round(-df_clean.loc[df_clean["Annulation"], "TotalPrice"].sum() / df_clean.loc[~df_clean["Annulation"], "TotalPrice"].sum() * 100, 1)
print("Part des annulations :", round(-df_clean.loc[df_clean["Annulation"], "TotalPrice"].sum() / df_clean.loc[~df_clean["Annulation"], "TotalPrice"].sum() * 100, 1), "%")

Part des annulations : 6.8 %


### 1.7 Bilan du nettoyage et sauvegarde

In [48]:
print("Lignes au départ :", df_raw.shape[0])
print("Lignes après nettoyage :", df_clean.shape[0])
print("Lignes supprimées au total :", df_raw.shape[0] - df_clean.shape[0])
print("Nombre de clients :", df_clean["CustomerID"].nunique())
print("Période : du", df_clean["InvoiceDate"].min(), "au", df_clean["InvoiceDate"].max())

df_clean.to_csv("../data/processed/online_retail_clean.csv", index=False)

Lignes au départ : 541909
Lignes après nettoyage : 401564
Lignes supprimées au total : 140345
Nombre de clients : 4371
Période : du 2010-12-01 08:26:00 au 2011-12-09 12:50:00


**Bilan du nettoyage :**
- On passe de **541 909** à **401 564 lignes**.
- Il reste **4 371 clients** différents.
- Période couverte : du 01/12/2010 au 09/12/2011.

Les données nettoyées sont enregistrées dans `data/processed/online_retail_clean.csv`.